In [1]:
from _src.Utils.CompositionLoader import CompositionExcelLoader
from _src.Composition.CompositionV2 import Composition
from _src.PhaseStability.TwoPhaseStabilityTestV3 import TwoPhaseStabilityTest
from _src.VLE.PhaseEquilibriumNewtonV2 import PhaseEquilibriumNewton
from _src.Utils.FluidPropertiesCalculatorV2 import FluidPropertiesCalculator


In [2]:
xl_loader = CompositionExcelLoader(r'C:\Users\user\Desktop\PVT_TSU\diss\prrzlm_mdt.xlsx')
comp_dict = xl_loader.load(header=True, sheet = 'to_model')

In [3]:
composition = Composition(comp_dict)
composition.evaluate_composition_data({'critical_temperature': 'kesler lee',
                                                    'critical_pressure': 'kesler lee',
                                                    'acentric_factor': 'riazi al sahhaf',
                                                    'critical_volume': 'hall yarborough',
                                                    'Kw': 'k watson',
                                                    'shift_parameter': 'jhaveri youngren'})

In [ ]:
composition.normalize_composition()

In [ ]:
composition.composition_data

In [ ]:
from _src.VLE.Flash import Flash
from _src.Utils.Conditions import Conditions

In [ ]:
conds = Conditions(190, 110 + 273.15)

In [ ]:
flash_obj = Flash(composition, conds)

In [ ]:
flash_res = flash_obj.calculate()

In [ ]:
flash_res

In [4]:
from _src.Experiments.DLE2 import DLE

In [5]:
dle_obj = DLE(composition, [190, 100, 60, 40, 20, 10, 5,], 220, 110)

In [6]:
dle_data = dle_obj.calculate()

2026-06-05 13:49:42,521 | _src.PhaseDiagram.new_methodv2 | INFO | Инициализация: T=383.14 K, P_min=0.0100, P_max=1000.0000
2026-06-05 13:49:42,522 | _src.PhaseDiagram.new_methodv2 | INFO | Запуск поиска давления насыщения
2026-06-05 13:49:42,522 | _src.PhaseDiagram.new_methodv2 | INFO | 🔍 Поиск bracket: start=[0.010, 1000.000]
2026-06-05 13:49:42,609 | _src.PhaseDiagram.new_methodv2 | INFO | Диапазон поиска: unstable=112.3136 bar -> stable=113.2901 bar
2026-06-05 13:49:42,610 | _src.PhaseDiagram.new_methodv2 | INFO | Старт бисекции с P=112.8018 bar, интервал=0.9766 bar
2026-06-05 13:49:42,754 | _src.PhaseDiagram.new_methodv2 | INFO | Интервал схлопнулся на итерации 21. Возврат P=113.2901 bar


[1.12670852 1.1199303  1.1339291  1.17081762 1.24893561 1.15113879
 1.14014084 1.08991115 1.06359051 1.        ]
[567.8687413  567.8687413  567.8687413  453.1306838  343.36876607
 247.97712636 159.43214628  77.85084606   0.           0.        ]


In [ ]:
dle_data

In [ ]:
import pandas as pd
import numpy as np
from typing import List

# Предполагаем, что FlashResult импортирован
from _src.VLE.Flash import FlashResult 

def vectorize_dle_results(flash_results: List['FlashResult']) -> pd.DataFrame:
    """
    Преобразует список объектов FlashResult в векторизованный Pandas DataFrame.
    Скалярные свойства разворачиваются в колонки, составы остаются как dict.
    """
    records = []
    
    # 1. Динамически собираем все возможные ключи свойств из всех результатов
    # Это делает функцию устойчивой к изменениям в FluidPropertiesCalculator
    all_prop_keys = set()
    for res in flash_results:
        if res.vapor.properties:
            all_prop_keys.update(res.vapor.properties.keys())
        if res.liquid.properties:
            all_prop_keys.update(res.liquid.properties.keys())
            
    prop_keys = sorted(list(all_prop_keys)) 
    # Например: ['density', 'molar_density', 'molecular_weight', 'molar_volume', 'viscosity']

    # 2. Проходим по каждому шагу и формируем строку данных
    for res in flash_results:
        record = {
            # Базовые условия
            'pressure': res.pressure,
            'temperature': res.temperature,
            'is_two_phase': res.is_two_phase,
            
            # Мольные доли фаз (всегда есть, даже если 0.0)
            'vapor_mole_frac': res.vapor.mole_fraction,
            'liquid_mole_frac': res.liquid.mole_fraction,
            
            # Составы оставляем как есть (объекты dict), как вы и просили
            'vapor_composition': res.vapor.composition,
            'liquid_composition': res.liquid.composition,
        }
        
        # 3. Векторизуем свойства динамически
        for key in prop_keys:
            # Для газа: берем значение или np.nan, если словарь пуст или ключа нет
            v_prop = res.vapor.properties.get(key, np.nan) if res.vapor.properties else np.nan
            record[f'vapor_{key}'] = v_prop
            
            # Для жидкости: аналогично
            l_prop = res.liquid.properties.get(key, np.nan) if res.liquid.properties else np.nan
            record[f'liquid_{key}'] = l_prop
            
        records.append(record)
        
    # 4. Создаем DataFrame. Pandas автоматически определит типы:
    # float64 для чисел, bool для is_two_phase, object для словарей составов
    df = pd.DataFrame(records)
    
    # Опционально: сортируем по давлению по убыванию (стандарт для DLE)
    df = df.sort_values('pressure', ascending=False).reset_index(drop=True)
    
    return df


vectorize_dle_results(dle_data)

In [ ]:
dle_df = vectorize_dle_results(dle_data)

In [ ]:
fl_arr = dle_df['liquid_mole_frac'].to_numpy()

## манипуляции с составом

In [ ]:
composition.edit_bip_for_components(component1='C1', component2='C2', bip = 0)
composition.edit_bip_for_components(component1='C1', component2='C3', bip = 0)
composition.edit_bip_for_components(component1='C1', component2='iC4', bip = 0)
composition.edit_bip_for_components(component1='C1', component2='nC4', bip = 0)
composition.edit_bip_for_components(component1='C1', component2='nC5', bip = 0)
composition.edit_bip_for_components(component1='C1', component2='C6', bip = 0)

composition.edit_bip_for_components(component1='C2', component2='C2', bip = 0)
composition.edit_bip_for_components(component1='C2', component2='C3', bip = 0)
composition.edit_bip_for_components(component1='C2', component2='iC4', bip = 0)
composition.edit_bip_for_components(component1='C2', component2='nC4', bip = 0)
composition.edit_bip_for_components(component1='C2', component2='nC5', bip = 0)
composition.edit_bip_for_components(component1='C2', component2='C6', bip = 0)

composition.edit_bip_for_components(component1='C3', component2='C2', bip = 0)
composition.edit_bip_for_components(component1='C3', component2='C3', bip = 0)
composition.edit_bip_for_components(component1='C3', component2='iC4', bip = 0)
composition.edit_bip_for_components(component1='C3', component2='nC4', bip = 0)
composition.edit_bip_for_components(component1='C3', component2='nC5', bip = 0)
composition.edit_bip_for_components(component1='C3', component2='C6', bip = 0)

composition.edit_bip_for_components(component1='iC4', component2='C2', bip = 0)
composition.edit_bip_for_components(component1='iC4', component2='C3', bip = 0)
composition.edit_bip_for_components(component1='iC4', component2='iC4', bip = 0)
composition.edit_bip_for_components(component1='iC4', component2='nC4', bip = 0)
composition.edit_bip_for_components(component1='iC4', component2='nC5', bip = 0)
composition.edit_bip_for_components(component1='iC4', component2='C6', bip = 0)

composition.edit_bip_for_components(component1='nC4', component2='C2', bip = 0)
composition.edit_bip_for_components(component1='nC4', component2='C3', bip = 0)
composition.edit_bip_for_components(component1='nC4', component2='iC4', bip = 0)
composition.edit_bip_for_components(component1='nC4', component2='nC4', bip = 0)
composition.edit_bip_for_components(component1='nC4', component2='nC5', bip = 0)
composition.edit_bip_for_components(component1='nC4', component2='C6', bip = 0)

composition.edit_bip_for_components(component1='nC5', component2='C2', bip = 0)
composition.edit_bip_for_components(component1='nC5', component2='C3', bip = 0)
composition.edit_bip_for_components(component1='nC5', component2='iC4', bip = 0)
composition.edit_bip_for_components(component1='nC5', component2='nC4', bip = 0)
composition.edit_bip_for_components(component1='nC5', component2='nC5', bip = 0)
composition.edit_bip_for_components(component1='nC5', component2='C6', bip = 0)

In [ ]:
composition.composition_data

In [ ]:
composition.T = 110 + 273.15

In [ ]:
condition_p = 274.4213
condition_t = 330 + 273.15

In [ ]:
phase_stab = TwoPhaseStabilityTest(composition, condition_p, condition_t)
phase_stab.calculate_phase_stability()
phase_stab.stable

In [ ]:
phase_stab.convergence_trivial_solution_l

In [ ]:
phase_stab.convergence_trivial_solution_v

In [ ]:
phase_stab.S_l

In [ ]:
phase_stab.S_v

In [ ]:
phase_stab.S_v_rounded

In [ ]:
phase_equil = PhaseEquilibriumNewton(composition, condition_p, condition_t, phase_stab.k_values_for_flash)
flash_data = phase_equil.find_solve_loop()

In [ ]:
flash_data

In [ ]:
T_min = 310
T_max = 340
p_min = 260
p_max = 275
import numpy as np
import matplotlib.pyplot as plt


T_minK = float(T_min) + 273.15
T_maxK = float(T_max) + 273.15
p_minBar = float(p_min)
p_maxBar = float(p_max)    

p_arr = np.linspace(p_minBar, p_maxBar, 30)
T_arr = np.linspace(T_minK, T_maxK, 30)

#composition = composition.new_composition(composition.composition, deep_copy=True)

result_p_arr = []
result_T_arr = []
result_c_arr = []

for p in p_arr:
    for T in T_arr:
        composition.T = T
        stab_test_obj = TwoPhaseStabilityTest(composition=composition, p=p, t=T)
        stab_test_obj.calculate_phase_stability()

        if stab_test_obj.stable:
            result_c_arr.append(0.0)
        else:
            result_c_arr.append(1.0)

        result_p_arr.append(p)
        result_T_arr.append(T - 273.15)

plt.scatter(x=result_T_arr, y=result_p_arr, c=result_c_arr)
plt.xlabel('T, C')
plt.ylabel('P, bar')
plt.title('Фазовая диаграмма только через тест стабильности')
plt.show()

In [ ]:
T_min = 310
T_max = 340
p_min = 175
p_max = 195
import numpy as np
import matplotlib.pyplot as plt


T_minK = float(T_min) + 273.15
T_maxK = float(T_max) + 273.15
p_minBar = float(p_min)
p_maxBar = float(p_max)    

p_arr = np.linspace(p_minBar, p_maxBar, 30)
T_arr = np.linspace(T_minK, T_maxK, 30)

composition = composition.new_composition(composition.composition, deep_copy=True)

result_p_arr = []
result_T_arr = []
result_c_arr = []

for p in p_arr:
    for T in T_arr:
        composition.T = T
        stab_test_obj = TwoPhaseStabilityTest(composition=composition, p=p, t=T)
        stab_test_obj.calculate_phase_stability()

        if stab_test_obj.stable:
            result_c_arr.append(0.0)
        else:
            result_c_arr.append(1.0)

        result_p_arr.append(p)
        result_T_arr.append(T - 273.15)

plt.scatter(x=result_T_arr, y=result_p_arr, c=result_c_arr)
plt.xlabel('T, C')
plt.ylabel('P, bar')
plt.title('Фазовая диаграмма только через тест стабильности')
plt.show()

In [ ]:
phase_equil = PhaseEquilibriumNewton(composition, condition_p, condition_t, phase_stab.k_values_for_flash)
flash_data = phase_equil.find_solve_loop()

### Проблема: FluidProperties от Дениса будто некорректный, надо проверить на старом модуле

In [ ]:
fluid_props = FluidPropertiesCalculator(composition=flash_data['xi_l'], composition_properties= composition.composition_data,
                                        p= condition_p, T=condition_t, eos_object = phase_equil.eos_liquid)

In [ ]:
fluid_props.density

In [ ]:
fluid_props.viscosity

In [ ]:
fluid_props.z_shift

## Работа через интерфейс  Model

In [ ]:
from _src.CompositionalModel.CompositionalModelV2 import CompositionalModel
from _src.Utils.Conditions import Conditions

In [ ]:
model = CompositionalModel(composition)

In [ ]:
conds = Conditions(100, 383)

In [ ]:
model.flash(110,373)

In [ ]:
model.flash(111, 373)

In [ ]:
model.result_store_object._results[0]

In [ ]:
model.result_store_object.get_by_module('Flash')

In [ ]:
from _src.Utils.Export2 import ModelJSONDB

In [ ]:
model_jsn = ModelJSONDB()

In [ ]:
model_jsn.export('KRSNL_1', 'KRASNOLEN_TEST', composition.composition, composition.composition_data,
                 composition.eos_name, model.result_store_object._results, field='KRSNLN')

In [ ]:
model_jsn.save()

## Новый переписанный модуль по расчету ЗhaseEnv

In [ ]:
from _src.PhaseDiagram.new_methodv2 import SaturationPressure

In [ ]:
sat_p = SaturationPressure(composition, t_K= 110+273.15, p_max_bar=300)

In [ ]:
sat_p.sp_convergence_loop()

In [ ]:
sat_p.p_i

In [ ]:
import numpy as np

t_arr = np.linspace(273.15, 550+273.15, 20)

for t in t_arr:
    sat_p = SaturationPressure(composition=composition,
                               t_K=t, p_max_bar= 500)
    res_sp = sat_p.sp_convergence_loop()
    res_dp = sat_p.dp_convergence_loop()

    print('====')
    print(f'RESULT: PSAT:{res_sp}, PDEW:{res_dp}, T:{t-273.15}')
    print('====')


## Фазовая

In [ ]:
from _src.PhaseDiagram.new_phase_diag import PhaseDiagram

In [ ]:
phase_new = PhaseDiagram(composition, 1000, 0, 800, 20)

In [ ]:
phase_new.calc_phase_diagram()

phase_new.plot()

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')


# Пример состава
comp = Composition({'C1': 0.4, 'C2': 0.2, 'nC4': 0.1, 'C6': 0.3})
comp.evaluate_composition_data({'critical_temperature': 'kesler lee',
                                                    'critical_pressure': 'kesler lee',
                                                    'acentric_factor': 'riazi al sahhaf',
                                                    'critical_volume': 'hall yarborough',
                                                    'Kw': 'k watson',
                                                    'shift_parameter': 'jhaveri youngren'})


# Расчёт от 0 до 200°C с шагом 10°C, P_max = 500 bar
phase_diag = PhaseDiagram(comp, p_max_bar=500, t_min_c=0, t_max_c=200, t_step_c=10)
phase_diag.calc_phase_diagram()

# Экспорт и визуализация
df = phase_diag.to_dataframe()
print(df.head())
phase_diag.plot(save_path="phase_envelope.png")

## через флеш

In [ ]:
from _src.PhaseDiagram.new_method import SaturationPressureFromFlash

In [ ]:
sat_p_from_flash = SaturationPressureFromFlash(composition, 110+273.15)

In [ ]:
sat_p_from_flash.loop()